# 🧪 معمل التجارب: بناء مسار RAG باستخدام LCEL ومشغل الربط (|) (Lecture 28)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في المحاضرة 28 نتعلم الترقية الكبرى في لانج تشين: الانتقال من السلاسل الجاهزة القديمة إلى **LCEL (LangChain Expression Language)**.
بدل الدوال الجاهزة، نستخدم مشغل الربط `|` (The Pipe Operator) عشان نركب مسار الـ RAG زي مكعبات الليجو:
1. المسترجع يجيب النصوص ⬅️ `retriever | format_docs`
2. السؤال يمر زي ما هو ⬅️ `RunnablePassthrough()`
3. يتجمعوا في القالب ⬅️ `| prompt`
4. يروح للموديل ⬅️ `| llm`
5. يخرج كنص صافي ⬅️ `| StrOutputParser()`

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحديد المسارات المركزية
**الفكرة ببساطة:**
بنستورد مكونات LCEL الأساسية: `RunnablePassthrough` و `StrOutputParser` ونحدد مسار حفظ البيانات المركزي لضمان بقاء مجلدات المشروع منظمة ونظيفة.

In [ ]:
# 1. import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized project data paths
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
docs_dir = str(data_dir / "chroma_sample_data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# import langchain components
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("✅ LCEL components and libraries imported successfully!")
print(f"📁 Database Path: {db_dir}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتحديد المسار بنفسك هنا):



### 2. قراءة المستندات وبناء مسترجع ChromaDB
**الفكرة ببساطة:**
بنجهز قاعدة المتجهات ChromaDB ونتأكد إنها محملة بالمستندات، ونجهز الـ `Retriever` ليسترجع أفضل 3 قطع مطابقة دلالياً.

In [ ]:
# initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# connect or create chroma vector store
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)

# if database is empty, load and ingest sample documents
if vector_store._collection.count() == 0:
    loader = DirectoryLoader(docs_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    raw_docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    chunked = splitter.split_documents(raw_docs)
    vector_store = Chroma.from_documents(chunked, embeddings, persist_directory=db_dir, collection_name="rag_collection")

# create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"✅ ChromaDB connected! Total stored vectors: {vector_store._collection.count()}")
print(f"🔍 Retriever ready with k={retriever.search_kwargs['k']}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تجهيز المسترجع من قاعدة Chroma بنفسك هنا):



### 3. دالة دمج المستندات (`format_docs`)
**الفكرة ببساطة:**
المسترجع بيرجع قايمة من كائنات `Document`. الدالة دي بتمر عليهم وتاخد نص كل قطعة (`doc.page_content`) وتدمجهم في نص واحد مفصول بسطرين عشان يتحط في خانة `{context}` ببساطة وسلاسة.

In [ ]:
# helper function to combine retrieved document chunks into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# test format_docs on sample retriever output
test_docs = retriever.invoke("What is deep learning?")
print(f"✅ Formatted {len(test_docs)} retrieved chunks into single context:")
print(format_docs(test_docs)[:140], "...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة دالة format_docs واختبارها بنفسك هنا):



### 4. إعداد قالب التوجيه (Prompt) ونموذج اللغة (Groq LLM)
**الفكرة ببساطة:**
بنصمم الـ Prompt ببساطة بقالب يحتوي على خانتين أساسيتين: `{context}` للسياق المسترجع، و `{question}` لسؤال المستخدم، وبنشغل موديل Groq السريع.

In [ ]:
# prompt template enforcing strict context grounding
template = """You are a helpful AI assistant.
Answer the question using ONLY the provided context.
If you don't know the answer, say "I don't know."
Keep the answer concise.

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# initialize fast Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

print("✅ Prompt template and Groq LLM initialized!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إعداد الـ Prompt والموديل بنفسك هنا):



### 5. بناء مسار الـ RAG بمشغل الربط (`|`)
**الفكرة ببساطة:**
هنا عبقرية LCEL! بنربط الأجزاء في سطر واحد بدون أي كود وسيط معقد:
القاموس بينفذ استرجاع السياق وتمرير السؤال بالتوازي، وبعدين يمررهم للـ Prompt، وبعدين للـ LLM، وبعدين لمحول النصوص الصافي `StrOutputParser()`.

In [ ]:
# build full lcel rag chain using pipe operator
rag_chain_lcel = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ LCEL RAG chain created successfully!")
print(f"Chain structure: {rag_chain_lcel}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء مسار LCEL بمشغل الربط بنفسك هنا):



### 6. تجربة الاستعلام العادي والبث المباشر (Streaming)
**الفكرة ببساطة:**
بفضل LCEL:
1. نقدر ننادي السلسلة بسؤال نصي مباشر `.invoke("سؤالك")`.
2. السلسلة بتدعم البث اللحظي للكلمات `.stream("سؤالك")` تلقائياً بدون كتابة أي كود إضافي!

In [ ]:
# test 1: invoke pipeline with a direct string query
query = "What is deep learning and how does it differ from traditional machine learning?"
answer = rag_chain_lcel.invoke(query)
print(f"❓ Question: {query}\n")
print(f"💬 Answer:\n{answer}\n")

print("=" * 60)
# test 2: real-time token-by-token streaming
stream_query = "What are CNNs best used for?"
print(f"🌊 Streaming answer for: '{stream_query}'\n")

for chunk in rag_chain_lcel.stream(stream_query):
    print(chunk, end="", flush=True)
print("\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استدعاء السلسلة وتجربة الـ streaming بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 28):
1. **LCEL ورمز `|`:** الطريقة الحديثة والقياسية في LangChain، أكثر مرونة وشفافية من السلاسل القديمة.
2. **`RunnablePassthrough`:** بيمرر السؤال الأصلي بدون تغيير للـ Prompt بالتوازي مع استرجاع السياق.
3. **`StrOutputParser`:** بيحول مخرجات النموذج من كائن `AIMessage` لنص عادي (`str`) مباشرة.
4. **التدفق التلقائي (`Streaming`):** كل سلاسل LCEL بتدعم `.stream()` فورياً بدون إعدادات معقدة.